# Knee Osteoarthritis: Is the Low 5-Class Accuracy Caused by Similar Neighbouring Grades?
### A binary re-analysis – KL0 (no OA) vs KL1–KL4 (OA)
**Course:** XBDS3024N – Image Processing and Computer Vision  |  Companion notebook to `IPCV_Knee_OA_4Models.ipynb`

**Why this notebook?** In the main notebook the best five-grade models reached only ≈47–55 % test accuracy, and most errors were between *neighbouring* grades (e.g. KL1 ↔ KL2, KL4 → KL3). This notebook tests the explanation behind that result:

> **Hypothesis:** the models are not "bad at seeing knees"; the five-grade task is hard because neighbouring KL grades differ by small, gradual changes (slightly narrower joint space, a slightly bigger osteophyte), so their X-rays look almost the same. If that is true, then:
> 1. merging grades into a coarser **binary** task should give a clearly higher accuracy *and* a higher chance-corrected agreement (Cohen's κ);
> 2. many of the five-grade models' errors should disappear when their predictions are merged into the binary classes (the errors stay inside a merged group);
> 3. the binary errors that remain should sit at the grades next to the class boundary, and move when the boundary moves;
> 4. pairs of grades that are **far apart** (KL0 vs KL4) should be almost perfectly separable, while **adjacent** pairs (KL0 vs KL1) should be much harder to separate.

**Design (kept identical to the main notebook, so the comparison is fair):** same patient-disjoint splits, same 224 × 224 grayscale images, baseline preprocessing (CLAHE did not help any model in the main study), same HOG and frozen ImageNet CNN features, the same four models with the same hyperparameter grids, selection on **validation**, and the test set used only for reporting.

| Task | Class 0 | Class 1 | Purpose |
|---|---|---|---|
| **A (main request)** | KL0 – no OA | KL1 + KL2 + KL3 + KL4 – any OA sign | Does a coarse "healthy vs OA" question become easy? |
| **B (sensitivity check)** | KL0 + KL1 – no / doubtful OA | KL2 + KL3 + KL4 – definite OA | Same question with the "doubtful" KL1 moved to the healthy side (the usual epidemiological definition of radiographic OA is KL ≥ 2) |

Task B shows whether the remaining errors of task A come from one specific grade – KL1, which is "doubtful" by definition.

**Note:** experimental coursework model – not a medical diagnostic tool.

### Contents
1. Setup · 2. Data and binary labels · 3. Preprocessing and features · 4. Imbalance handling · 5. Models and experiment framework · 6. Binary results · 7. Comparison with the five-class models · 8. Why neighbouring grades are hard: separability and visual similarity · 9. Conclusion

---
## 1. Setup and configuration
Place this notebook in the **same folder** as the main notebook: it then re-uses the cached HOG and CNN features in `cache/` (no re-extraction) and reads the five-class results from `results/`. New results are written to `results_binary/`.

In [ ]:
import json
import os
import random
import time
from abc import ABC, abstractmethod
from pathlib import Path

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models

# ---- Paths (same defaults as the main notebook) -------------------------------------------------
DATA_DIR = Path(os.environ.get("KNEE_DATA_DIR",
                               r"C:\Users\ACER NITRO\Documents\IPCV\Assignment_IPCV_Knee\Knee_Dataset"))
CACHE_DIR = Path("cache")                    # shared with the main notebook
MAIN_RESULTS_DIR = Path("results")           # five-class results of the main notebook
RESULTS_DIR = Path("results_binary")         # results of this notebook
WEIGHTS_DIR = Path(os.environ.get("KERAS_WEIGHTS_DIR", "weights"))
RESULTS_DIR.mkdir(exist_ok=True)

# ---- Settings (identical to the main notebook) -------------------------------------------------
SEED = 42
IMAGE_SIZE = 224
GRADES = ["KL0", "KL1", "KL2", "KL3", "KL4"]
HOG_PARAMS = dict(orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))
SVM_C_GRID = [0.0003, 0.001, 0.003]
RF_N_TREES, RF_DEPTH_GRID, RF_MIN_LEAF = 300, [10, 20, None], 2
NEURAL_BACKBONES = ["MobileNetV2", "DenseNet121"]
HEAD_EPOCHS = int(os.environ.get("HEAD_EPOCHS", 100))
HEAD_PATIENCE, HEAD_BATCH, HEAD_LR = 10, 64, 1e-3
REUSE_SAVED_RESULTS = True

# ---- The two binary tasks: grade -> new class -----------------------------------------------------
TASKS = {
    "A": {"title": "KL0 vs KL1–4", "classes": ["No OA (KL0)", "OA (KL1–4)"], "mapping": [0, 1, 1, 1, 1]},
    "B": {"title": "KL0–1 vs KL2–4", "classes": ["No/doubtful OA (KL0–1)", "Definite OA (KL2–4)"], "mapping": [0, 0, 1, 1, 1]},
}

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("Dataset folder:", DATA_DIR, "| exists:", DATA_DIR.is_dir())
print("Feature cache :", CACHE_DIR.resolve(), "| exists:", CACHE_DIR.is_dir())
print("TensorFlow:", tf.__version__)

---
## 2. Data and binary labels
### 2.1 Loading the splits
As in the main notebook, each split folder has sub-folders `0`–`4` whose names are the KL grades. `KneeSplit` keeps the original grade; the binary label is derived from it by a `BinaryTask`, so the five-class and binary analyses always use exactly the same images in the same order.

In [ ]:
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp")


class KneeSplit:
    """Image paths and original KL grades (0–4) for one split."""

    def __init__(self, root: Path, split: str) -> None:
        self.root, self.split = Path(root), split
        self.paths: list[Path] = []
        self.labels: list[int] = []
        for class_dir in sorted((self.root / split).iterdir()):
            if class_dir.is_dir():
                for file in sorted(class_dir.iterdir()):
                    if file.suffix.lower() in IMAGE_EXTENSIONS:
                        self.paths.append(file)
                        self.labels.append(int(class_dir.name))

    def __len__(self) -> int:
        return len(self.paths)

    @property
    def grades(self) -> np.ndarray:
        return np.asarray(self.labels, dtype=np.int64)


class BinaryTask:
    """Maps the five KL grades to two classes."""

    def __init__(self, key: str, title: str, classes: list[str], mapping: list[int]) -> None:
        self.key, self.title, self.classes, self.mapping = key, title, classes, np.asarray(mapping)

    def __call__(self, grades: np.ndarray) -> np.ndarray:
        return self.mapping[np.asarray(grades)]

    def describe(self) -> str:
        groups = {c: [GRADES[g] for g in range(5) if self.mapping[g] == i] for i, c in enumerate(self.classes)}
        return " vs ".join(f"{c} = {'+'.join(g)}" for c, g in groups.items())


SPLITS = {s: KneeSplit(DATA_DIR, s) for s in ("train", "val", "test")}      # auto_test excluded (patient overlap)
TASK_OBJ = {k: BinaryTask(k, **v) for k, v in TASKS.items()}
print({s: len(sp) for s, sp in SPLITS.items()})
for t in TASK_OBJ.values():
    print(f"Task {t.key}: {t.describe()}")

### 2.2 Class distribution after merging
Merging changes the imbalance completely: in the five-class problem KL0 was the majority and KL4 a tiny minority (13 : 1). In task A, **OA becomes the majority** (≈60 %) and the imbalance is mild (≈1.5 : 1).

In [ ]:
rows = []
for t in TASK_OBJ.values():
    for s, sp in SPLITS.items():
        counts = np.bincount(t(sp.grades), minlength=2)
        rows.append({"Task": f"{t.key}: {t.title}", "Split": s, "Class 0 (n)": counts[0], "Class 1 (n)": counts[1], "Class 1 share": counts[1] / counts.sum(),
                     "Imbalance ratio": counts.max() / counts.min()})
dist = pd.DataFrame(rows)[["Task", "Split", "Class 0 (n)", "Class 1 (n)", "Class 1 share", "Imbalance ratio"]]
display(dist.style.format({"Class 1 share": "{:.1%}", "Imbalance ratio": "{:.2f}"}).hide(axis="index"))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
five = np.bincount(SPLITS["train"].grades, minlength=5)
axes[0].bar(GRADES, five, color=["#4C9F70", "#A3C45A", "#E3B23C", "#E07B39", "#C0392B"])
axes[0].set_title(f"Five-class (train): imbalance {five.max() / five.min():.1f} : 1")
for ax, t in zip(axes[1:], TASK_OBJ.values()):
    c = np.bincount(t(SPLITS["train"].grades), minlength=2)
    ax.bar(t.classes, c, color=["#4C9F70", "#C0392B"])
    ax.set_title(f"Task {t.key} (train): imbalance {c.max() / c.min():.2f} : 1")
    for i, v in enumerate(c):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom")
for ax in axes:
    ax.set_ylabel("Training images")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "binary_class_distribution.png", dpi=200)
plt.show()

### 2.3 What the two classes look like
Five random training X-rays per class for task A (top: KL0; bottom: two KL1 knees, then one KL2, KL3 and KL4). The two KL1 knees are hard to tell apart from the KL0 row by eye – the first sign that the boundary between "no OA" and "OA" passes through very similar images.

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(2, 5, figsize=(13, 5.6))
for row, cls in enumerate([0, 1]):
    grades = [0] * 5 if cls == 0 else [1, 1, 2, 3, 4]
    for col, g in enumerate(grades):
        idx = rng.choice(np.where(SPLITS["train"].grades == g)[0])
        axes[row, col].imshow(cv2.imread(str(SPLITS["train"].paths[idx]), cv2.IMREAD_GRAYSCALE), cmap="gray")
        axes[row, col].set_title(f"{TASK_OBJ['A'].classes[cls]}\n(original {GRADES[g]})", fontsize=9)
        axes[row, col].axis("off")
plt.suptitle("Task A – training examples of each class", y=1.01)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "binary_examples.png", dpi=200, bbox_inches="tight")
plt.show()

---
## 3. Preprocessing and feature extraction
Same pipeline as the main notebook: grayscale → 224 × 224 (area interpolation) → baseline preprocessing. Two representations:
* **HOG** (9 orientations, 16 × 16 cells, 2 × 2 blocks → 6,084 values) for the SVM and Random Forest;
* **frozen ImageNet CNN features** (global average pooling: MobileNetV2 1,280, DenseNet121 1,024 values) for the neural models.

The extractors use the same cache file names as the main notebook, so if this notebook sits in the same folder the features load instantly.

In [ ]:
class Preprocessor:
    """Read grayscale and resize (baseline preprocessing)."""
    name = "baseline"

    def __init__(self, image_size: int = IMAGE_SIZE) -> None:
        self.image_size = image_size

    def __call__(self, path: Path) -> np.ndarray:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise IOError(f"Could not read image: {path}")
        return cv2.resize(img, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)


PRE = Preprocessor()
_image_cache: dict = {}


def split_images(split: str) -> np.ndarray:
    """All images of one split as a uint8 array (N, 224, 224), loaded once."""
    if split not in _image_cache:
        _image_cache[split] = np.stack([PRE(p) for p in tqdm(SPLITS[split].paths, desc=f"Load {split}")])
    return _image_cache[split]


class FeatureExtractor(ABC):
    name = "base"

    @abstractmethod
    def extract_array(self, imgs: np.ndarray) -> np.ndarray: ...

    def params_tag(self) -> str:
        return ""

    def cache_file(self, split: str) -> Path:
        return CACHE_DIR / f"{self.name}_{PRE.name}_{PRE.image_size}{self.params_tag()}_{split}.npy"

    def features(self, split: str) -> np.ndarray:
        """Cached features of a split (same file names as the main notebook)."""
        file = self.cache_file(split)
        if file.exists():
            feats = np.load(file)
            if len(feats) == len(SPLITS[split]):
                return feats
        CACHE_DIR.mkdir(exist_ok=True)
        feats = self.extract_array(split_images(split)).astype(np.float32)
        np.save(file, feats)
        return feats


class HogExtractor(FeatureExtractor):
    name = "hog"

    def params_tag(self):
        return f"_o{HOG_PARAMS['orientations']}_c{HOG_PARAMS['pixels_per_cell'][0]}_b{HOG_PARAMS['cells_per_block'][0]}"

    def extract_array(self, imgs):
        return np.stack([hog(i, block_norm="L2-Hys", feature_vector=True, **HOG_PARAMS) for i in tqdm(imgs, desc="HOG")])


BACKBONES = {
    "MobileNetV2": (tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input,
                    "mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5"),
    "DenseNet121": (tf.keras.applications.DenseNet121, tf.keras.applications.densenet.preprocess_input,
                    "densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5"),
}


class CnnFeatureExtractor(FeatureExtractor):
    """Frozen ImageNet CNN: grey X-ray -> 3 channels -> global-average-pooled feature vector."""

    def __init__(self, backbone: str, batch: int = 16) -> None:
        self.backbone, self.batch, self.name = backbone, batch, f"{backbone}-features"

    def params_tag(self):
        return "_imagenet_gap"

    def extract_array(self, imgs):
        constructor, preprocess, weights_file = BACKBONES[self.backbone]
        kwargs = dict(include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), pooling="avg")
        try:
            base = constructor(weights="imagenet", **kwargs)
        except Exception:
            base = constructor(weights=None, **kwargs)
            base.load_weights(str(WEIGHTS_DIR / weights_file))
        forward = tf.function(lambda x: base(x, training=False),
                              input_signature=[tf.TensorSpec([None, IMAGE_SIZE, IMAGE_SIZE, 3], tf.float32)])
        out = []
        for i in tqdm(range(0, len(imgs), self.batch), desc=self.name):
            x = np.repeat(imgs[i:i + self.batch, ..., None].astype(np.float32), 3, axis=-1)
            out.append(forward(tf.constant(preprocess(x))).numpy())
        return np.concatenate(out)


EXTRACTORS = {"HOG": HogExtractor(), **{b: CnnFeatureExtractor(b) for b in NEURAL_BACKBONES}}
FEATURES = {}
for rep, ext in EXTRACTORS.items():
    t0 = time.time()
    FEATURES[rep] = {s: ext.features(s) for s in SPLITS}
    print(f"{rep:<12} train {FEATURES[rep]['train'].shape} | val {FEATURES[rep]['val'].shape} | "
          f"test {FEATURES[rep]['test'].shape}  ({time.time() - t0:.1f} s)")
GRADES_OF = {s: SPLITS[s].grades for s in SPLITS}

---
## 4. Imbalance handling
The binary tasks are only mildly imbalanced (task A ≈1.5 : 1, task B ≈1.4 : 1), so two conditions are compared for every model:
* **None** – train on the natural distribution;
* **Class weighting** – weight each class by N / (2 × n_class) (scikit-learn `"balanced"`, weighted cross-entropy for the networks), the technique that worked best for the neural models in the main study.

Oversampling is not needed at this level of imbalance; keeping the comparison small also keeps the notebook fast.

In [ ]:
class ImbalanceStrategy:
    def __init__(self, label: str, weighted: bool) -> None:
        self.label, self.weighted = label, weighted

    def class_weight(self, y: np.ndarray) -> dict | None:
        if not self.weighted:
            return None
        counts = np.bincount(y, minlength=2)
        return {c: len(y) / (2 * n) for c, n in enumerate(counts)}


STRATEGIES = [ImbalanceStrategy("none", False), ImbalanceStrategy("class weighting", True)]
for t in TASK_OBJ.values():
    print(f"Task {t.key} class weights:", {t.classes[c]: round(w, 2) for c, w in STRATEGIES[1].class_weight(t(GRADES_OF['train'])).items()})

---
## 5. Models and experiment framework
The same four models as the main notebook, with the same settings; hyperparameters (SVM `C`, Random Forest depth, network epoch) are chosen on **validation macro-F1**.

Binary metrics reported: accuracy, balanced accuracy, **sensitivity** (recall of class 1, OA), **specificity** (recall of class 0), precision, F1, macro-F1, **ROC-AUC**, Cohen's **κ** (agreement corrected for chance), the confusion matrix, and the majority-class baseline.

In [ ]:
class BinaryEvaluator:
    """Binary metrics from labels, predictions and class-1 scores."""

    @staticmethod
    def evaluate(y: np.ndarray, pred: np.ndarray, score: np.ndarray) -> dict:
        return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
                "sensitivity": recall_score(y, pred, pos_label=1, zero_division=0),
                "specificity": recall_score(y, pred, pos_label=0, zero_division=0),
                "precision": precision_score(y, pred, pos_label=1, zero_division=0),
                "f1": f1_score(y, pred, pos_label=1, zero_division=0),
                "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
                "roc_auc": roc_auc_score(y, score), "kappa": cohen_kappa_score(y, pred),
                "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist()}


EVAL = BinaryEvaluator()


class BinaryExperiment(ABC):
    """One model + one imbalance condition on one binary task. Results are saved and re-loaded on re-runs."""

    family = "base"
    representation = "HOG"

    def __init__(self, task: BinaryTask, strategy: ImbalanceStrategy) -> None:
        self.task, self.strategy = task, strategy
        self.exp_id = f"{task.key}-{self.family.replace(' ', '')}-{strategy.label.replace(' ', '_')}"
        self.out_file = RESULTS_DIR / f"{self.exp_id}.json"

    @abstractmethod
    def fit_and_score(self, X: dict, y: dict, weights: dict | None) -> tuple[dict, dict]:
        """Train on X['train'] (choosing settings on validation); return class-1 scores per split and extra info."""

    def run(self) -> dict:
        if REUSE_SAVED_RESULTS and self.out_file.exists():
            return json.loads(self.out_file.read_text())
        t0 = time.time()
        X = FEATURES[self.representation]
        y = {s: self.task(GRADES_OF[s]) for s in SPLITS}
        scores, extra = self.fit_and_score(X, y, self.strategy.class_weight(y["train"]))
        res = {"id": self.exp_id, "task": self.task.key, "model": self.family, "imbalance": self.strategy.label, **extra,
               "majority_baseline_test_accuracy": float(max(np.mean(y["test"]), 1 - np.mean(y["test"]))),
               "run_time_min": (time.time() - t0) / 60}
        for s in SPLITS:
            res[s] = EVAL.evaluate(y[s], (scores[s] >= 0.5).astype(int), scores[s])
        res["test_scores"] = scores["test"].round(5).tolist()
        self.out_file.write_text(json.dumps(res, indent=1, default=float))
        return res


def softmax2(decision: np.ndarray) -> np.ndarray:
    """LinearSVC decision value -> score in [0, 1] (logistic); threshold 0.5 = decision 0."""
    return 1 / (1 + np.exp(-decision))


class SvmBinary(BinaryExperiment):
    family = "SVM"

    def fit_and_score(self, X, y, weights):
        scaler = StandardScaler().fit(X["train"])
        Xs = {s: scaler.transform(X[s]) for s in X}
        best = None
        for C in SVM_C_GRID:
            model = LinearSVC(C=C, class_weight=weights, dual="auto", max_iter=10000, random_state=SEED).fit(Xs["train"], y["train"])
            f1 = f1_score(y["val"], model.predict(Xs["val"]), average="macro")
            if best is None or f1 > best[0]:
                best = (f1, C, model)
        _, C, model = best
        return {s: softmax2(model.decision_function(Xs[s])) for s in Xs}, {"best_params": {"C": C}}


class RandomForestBinary(BinaryExperiment):
    family = "Random Forest"

    def fit_and_score(self, X, y, weights):
        best = None
        for depth in RF_DEPTH_GRID:
            model = RandomForestClassifier(n_estimators=RF_N_TREES, max_depth=depth, min_samples_leaf=RF_MIN_LEAF,
                                           max_features="sqrt", class_weight="balanced_subsample" if weights else None,
                                           n_jobs=-1, random_state=SEED).fit(X["train"], y["train"])
            f1 = f1_score(y["val"], model.predict(X["val"]), average="macro")
            if best is None or f1 > best[0]:
                best = (f1, depth, model)
        _, depth, model = best
        return {s: model.predict_proba(X[s])[:, 1] for s in X}, {"best_params": {"max_depth": depth}}


class MacroF1Stopping(tf.keras.callbacks.Callback):
    """Early stopping on validation macro-F1; keeps the best epoch's weights."""

    def __init__(self, X_val, y_val, patience):
        super().__init__()
        self.X_val, self.y_val, self.patience = X_val, y_val, patience
        self.best_f1, self.best_epoch, self.best_weights, self.wait = -1.0, 0, None, 0

    def on_epoch_end(self, epoch, logs=None):
        pred = self.model.predict(self.X_val, verbose=0).argmax(1)
        f1 = f1_score(self.y_val, pred, average="macro")
        if f1 > self.best_f1:
            self.best_f1, self.best_epoch, self.best_weights, self.wait = f1, epoch + 1, self.model.get_weights(), 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


class NeuralBinary(BinaryExperiment):
    """Frozen ImageNet features + the same classifier head as the main notebook (2 outputs)."""

    def __init__(self, task, strategy, backbone: str) -> None:
        self.family = self.representation = backbone
        super().__init__(task, strategy)

    def fit_and_score(self, X, y, weights):
        tf.keras.utils.set_random_seed(SEED)
        model = models.Sequential([layers.Input(shape=(X["train"].shape[1],)), layers.BatchNormalization(),
                                   layers.Dropout(0.2), layers.Dense(256, activation="relu"), layers.Dropout(0.5),
                                   layers.Dense(2, activation="softmax")])
        model.compile(optimizer=tf.keras.optimizers.Adam(HEAD_LR), loss="sparse_categorical_crossentropy")
        stopper = MacroF1Stopping(X["val"], y["val"], HEAD_PATIENCE)
        sample_weight = None if weights is None else np.array([weights[c] for c in y["train"]], dtype=np.float32)
        model.fit(X["train"], y["train"], sample_weight=sample_weight, epochs=HEAD_EPOCHS, batch_size=HEAD_BATCH,
                  shuffle=True, callbacks=[stopper], verbose=0)
        return {s: model.predict(X[s], verbose=0)[:, 1] for s in X}, {"best_epoch": stopper.best_epoch}


def make_experiments(task: BinaryTask) -> list[BinaryExperiment]:
    exps = []
    for strategy in STRATEGIES:
        exps += [SvmBinary(task, strategy), RandomForestBinary(task, strategy)]
        exps += [NeuralBinary(task, strategy, b) for b in NEURAL_BACKBONES]
    return exps

---
## 6. Binary results
### 6.1 Train every model on both tasks
2 tasks × 4 models × 2 imbalance conditions = 16 runs. Saved runs are re-loaded, so re-running the notebook is fast.

In [ ]:
RESULTS = {}
for task in TASK_OBJ.values():
    for exp in make_experiments(task):
        res = exp.run()
        RESULTS[res["id"]] = res
        t = res["test"]
        print(f"{res['id']:<38} val macro-F1 {res['val']['macro_f1']:.3f} | test acc {t['accuracy']:.3f} "
              f"bal.acc {t['balanced_accuracy']:.3f} AUC {t['roc_auc']:.3f} κ {t['kappa']:.3f}  ({res['run_time_min']:.1f} min)")

### 6.2 Results table and best configuration per model
The best imbalance condition of each model is chosen on **validation macro-F1** (as in the main notebook). Shaded rows are the selected configurations.

In [ ]:
def result_row(r: dict) -> dict:
    t = r["test"]
    return {"Task": TASK_OBJ[r["task"]].title, "Model": r["model"], "Imbalance": r["imbalance"], "Experiment": r["id"],
            "Val macro-F1": r["val"]["macro_f1"], "Train acc.": r["train"]["accuracy"], "Test acc.": t["accuracy"],
            "Balanced acc.": t["balanced_accuracy"], "Sensitivity": t["sensitivity"], "Specificity": t["specificity"],
            "Precision": t["precision"], "F1": t["f1"], "ROC-AUC": t["roc_auc"], "Kappa": t["kappa"],
            "Majority baseline": r["majority_baseline_test_accuracy"]}


table = pd.DataFrame([result_row(r) for r in RESULTS.values()])
BEST = {}                                               # (task, model) -> experiment id
for (task, model), grp in table.groupby(["Task", "Model"], sort=False):
    BEST[(task, model)] = grp.loc[grp["Val macro-F1"].idxmax(), "Experiment"]
table["Selected"] = table["Experiment"].isin(BEST.values())
table.to_csv(RESULTS_DIR / "binary_results.csv", index=False)
num = [c for c in table.columns if table[c].dtype.kind == "f"]
display(table.style.format({c: "{:.3f}" for c in num}).hide(axis="index")
        .apply(lambda r: ["background-color: #e2efda" if r["Selected"] else "" for _ in r], axis=1))

### 6.3 Confusion matrices and ROC curves (selected configurations)
The confusion matrices show *which* class is missed; the ROC curve shows how well each model ranks OA above no-OA across all thresholds (AUC 0.5 = guessing, 1.0 = perfect).

In [ ]:
model_order = ["SVM", "Random Forest"] + NEURAL_BACKBONES
for task in TASK_OBJ.values():
    fig, axes = plt.subplots(1, 5, figsize=(21, 4.2), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.25]})
    y_test = task(GRADES_OF["test"])
    for ax, model in zip(axes[:4], model_order):
        r = RESULTS[BEST[(task.title, model)]]
        cm = np.asarray(r["test"]["confusion_matrix"])
        ax.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i, j]}\n({cm[i, j] / cm[i].sum():.0%})", ha="center", va="center",
                        color="white" if cm[i, j] / cm[i].sum() > 0.5 else "black")
        ax.set_xticks([0, 1], ["pred " + c.split(" (")[0] for c in task.classes], fontsize=8)
        ax.set_yticks([0, 1], ["true " + c.split(" (")[0] for c in task.classes], fontsize=8)
        ax.set_title(f"{model} ({r['imbalance']})\nacc {r['test']['accuracy']:.3f}", fontsize=9)
        fpr, tpr, _ = roc_curve(y_test, r["test_scores"])
        axes[4].plot(fpr, tpr, label=f"{model} (AUC {r['test']['roc_auc']:.3f})")
    axes[4].plot([0, 1], [0, 1], "k--", lw=1)
    axes[4].set_xlabel("1 – specificity")
    axes[4].set_ylabel("Sensitivity")
    axes[4].set_title("ROC curves (test)", fontsize=9)
    axes[4].legend(fontsize=8)
    plt.suptitle(f"Task {task.key}: {task.title}", y=1.03)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"binary_task{task.key}_cm_roc.png", dpi=200, bbox_inches="tight")
    plt.show()

### 6.4 Where do the remaining binary errors come from?
For task A, each test image keeps its original grade, so the errors can be traced back: which original grades end up on the wrong side of the KL0 | KL1–4 boundary? If the hypothesis is right, the errors should be concentrated in **KL0 and KL1** – the two grades that sit right next to the boundary – and almost absent for KL3–KL4.

In [ ]:
rows = []
for task in TASK_OBJ.values():
    y_test = task(GRADES_OF["test"])
    for model in model_order:
        r = RESULTS[BEST[(task.title, model)]]
        pred = (np.asarray(r["test_scores"]) >= 0.5).astype(int)
        for g in range(5):
            mask = GRADES_OF["test"] == g
            rows.append({"Task": task.title, "Model": model, "Original grade": GRADES[g],
                         "Correct side of boundary": (pred[mask] == y_test[mask]).mean()})
by_grade = pd.DataFrame(rows)
pivot = by_grade.pivot_table(index=["Task", "Model"], columns="Original grade", values="Correct side of boundary", sort=False)
display(pivot.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0.3, vmax=1.0))

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), sharey=True)
for ax, task in zip(axes, TASK_OBJ.values()):
    sub = by_grade[by_grade["Task"] == task.title]
    for k, model in enumerate(model_order):
        vals = sub[sub["Model"] == model]["Correct side of boundary"].values
        ax.bar(np.arange(5) + (k - 1.5) * 0.2, vals, 0.2, label=model)
    boundary = np.where(np.diff(task.mapping) != 0)[0][0] + 0.5
    ax.axvline(boundary, color="red", ls="--", lw=1.5)
    ax.text(boundary, 1.03, " class boundary", color="red", fontsize=8)
    ax.set_xticks(range(5), GRADES)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"Task {task.key}: {task.title} – share of each grade classified correctly", fontsize=10)
axes[0].set_ylabel("Correct side of the boundary")
axes[0].legend(fontsize=8, ncol=2, loc="lower left")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "binary_errors_by_grade.png", dpi=200)
plt.show()

---
## 7. Comparison with the five-class models
Three numbers are compared for each model:
1. **Five-class** – the best five-grade configuration of the main notebook (selected on validation, from `results/`);
2. **Five-class → binary** – the *same* five-grade predictions, merged into the two task classes after prediction (no retraining). This isolates how many of the five-class errors were only confusions **between neighbouring grades on the same side of the boundary**;
3. **Binary model** – the model trained directly on the binary task (§6).

Because a binary task has a higher chance level (50 % or the majority share) than a five-class task (20 % or 39 %), raw accuracy alone would flatter the binary task. Two chance-aware measures are therefore added: **Cohen's κ** (agreement beyond what the class frequencies would give by chance) and **balanced accuracy above chance** (balanced accuracy − 1 / number of classes).

In [ ]:
def load_five_class() -> dict:
    """Best five-class experiment per model (highest validation macro-F1) from the main notebook's results."""
    runs = {}
    for f in sorted(MAIN_RESULTS_DIR.glob("E*/metrics.json")):
        r = json.loads(f.read_text())
        if r.get("id") in [f"E{i}" for i in range(1, 21)] and "y_pred_test" in r:
            runs[r["id"]] = r
    best = {}
    for model in model_order:
        cand = [r for r in runs.values() if r.get("family") == model]
        if cand:
            best[model] = max(cand, key=lambda r: r["val"]["macro_f1"])
    return best


FIVE = load_five_class()
if not FIVE:
    print(f"No five-class results found in {MAIN_RESULTS_DIR.resolve()} – run the main notebook first or copy its results/ folder here.")
else:
    y5 = GRADES_OF["test"]
    rows = []
    for model, r in FIVE.items():
        p5 = np.asarray(r["y_pred_test"])
        assert len(p5) == len(y5), "Test order differs from the main notebook"
        rows.append({"Model": model, "Setting": "Five-class (main notebook)", "Experiment": r["id"],
                     "Accuracy": accuracy_score(y5, p5), "Balanced acc.": balanced_accuracy_score(y5, p5),
                     "Bal. acc. above chance": balanced_accuracy_score(y5, p5) - 1 / 5,
                     "Kappa": cohen_kappa_score(y5, p5), "Chance accuracy (majority)": np.bincount(y5).max() / len(y5)})
        for task in TASK_OBJ.values():
            yb, pb = task(y5), task(p5)
            rows.append({"Model": model, "Setting": f"Five-class → {task.title} (merged after prediction)", "Experiment": r["id"],
                         "Accuracy": accuracy_score(yb, pb), "Balanced acc.": balanced_accuracy_score(yb, pb),
                         "Bal. acc. above chance": balanced_accuracy_score(yb, pb) - 1 / 2,
                         "Kappa": cohen_kappa_score(yb, pb), "Chance accuracy (majority)": max(yb.mean(), 1 - yb.mean())})
            b = RESULTS[BEST[(task.title, model)]]
            rows.append({"Model": model, "Setting": f"Binary model: {task.title}", "Experiment": b["id"],
                         "Accuracy": b["test"]["accuracy"], "Balanced acc.": b["test"]["balanced_accuracy"],
                         "Bal. acc. above chance": b["test"]["balanced_accuracy"] - 1 / 2,
                         "Kappa": b["test"]["kappa"], "Chance accuracy (majority)": b["majority_baseline_test_accuracy"]})
    comparison = pd.DataFrame(rows)
    comparison.to_csv(RESULTS_DIR / "five_vs_binary.csv", index=False)
    display(comparison.style.format({c: "{:.3f}" for c in comparison.columns if comparison[c].dtype.kind == "f"})
            .hide(axis="index").background_gradient(subset=["Accuracy", "Kappa"], cmap="Greens"))

    settings = ["Five-class (main notebook)"] + [f"Binary model: {t.title}" for t in TASK_OBJ.values()]
    colours = ["#9e9e9e", "#2a9d8f", "#264653"]
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
    for ax, metric in zip(axes, ["Accuracy", "Kappa"]):
        for k, (setting, colour) in enumerate(zip(settings, colours)):
            vals = [comparison[(comparison["Model"] == m) & (comparison["Setting"] == setting)][metric].values[0] for m in FIVE]
            xs = np.arange(len(FIVE)) + (k - 1) * 0.27
            bars = ax.bar(xs, vals, 0.27, color=colour, label=setting.replace("Binary model: ", "Binary: "))
            ax.bar_label(bars, fmt="%.2f", fontsize=7)
            if metric == "Accuracy":                       # black dash = majority-class accuracy of that task
                chance = comparison[comparison["Setting"] == setting]["Chance accuracy (majority)"].iloc[0]
                for x in xs:
                    ax.plot([x - 0.12, x + 0.12], [chance, chance], color="black", lw=2)
        ax.set_xticks(range(len(FIVE)), list(FIVE))
        ax.set_ylim(0, 1.05)
        ax.set_title({"Accuracy": "Test accuracy (black dash = majority-class accuracy of that task)",
                      "Kappa": "Cohen's κ – agreement corrected for chance"}[metric], fontsize=10)
    axes[0].legend(fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "five_vs_binary.png", dpi=200)
    plt.show()

    # How many five-class errors were only "neighbour" errors that stay on the same side of the boundary?
    for model, r in FIVE.items():
        p5 = np.asarray(r["y_pred_test"])
        wrong = p5 != y5
        msg = [f"{model} ({r['id']}): {wrong.sum()} five-class errors"]
        for task in TASK_OBJ.values():
            same_side = (task(p5) == task(y5)) & wrong
            msg.append(f"{same_side.sum() / wrong.sum():.0%} stay inside the same {task.title} group")
        msg.append(f"{(np.abs(p5 - y5)[wrong] == 1).mean():.0%} are exactly one grade off")
        print(" | ".join(msg))

---
## 8. Why neighbouring grades are hard: separability and visual similarity
### 8.1 Pairwise grade separability
For every pair of KL grades, a simple linear classifier (standardised features → logistic regression, class-weighted) is trained on the training images of **only those two grades** and tested on the test images of the same two grades. The test **ROC-AUC** measures how separable the two grades are with the features the models actually use (0.5 = indistinguishable, 1.0 = perfectly separable). This is done for HOG and for the best CNN features.

**Expected if the hypothesis holds:** AUC rises steadily with the distance between the grades – adjacent pairs (on the first off-diagonal) are the hardest.

In [ ]:
def pairwise_auc(rep: str) -> np.ndarray:
    X, auc = FEATURES[rep], np.full((5, 5), np.nan)
    for i in range(5):
        for j in range(i + 1, 5):
            tr = np.isin(GRADES_OF["train"], [i, j])
            te = np.isin(GRADES_OF["test"], [i, j])
            clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, class_weight="balanced", max_iter=3000))
            clf.fit(X["train"][tr], (GRADES_OF["train"][tr] == j).astype(int))
            auc[i, j] = auc[j, i] = roc_auc_score((GRADES_OF["test"][te] == j).astype(int), clf.predict_proba(X["test"][te])[:, 1])
    return auc


pair_file = RESULTS_DIR / "pairwise_auc.json"
if REUSE_SAVED_RESULTS and pair_file.exists():
    PAIR_AUC = {k: np.asarray(v, dtype=float) for k, v in json.loads(pair_file.read_text()).items()}
else:
    PAIR_AUC = {}
    for rep in ["HOG", "DenseNet121"]:
        t0 = time.time()
        PAIR_AUC[rep] = pairwise_auc(rep)
        print(f"{rep}: done in {time.time() - t0:.0f} s")
    pair_file.write_text(json.dumps({k: np.nan_to_num(v, nan=-1).tolist() for k, v in PAIR_AUC.items()}))
PAIR_AUC = {k: np.where(v < 0, np.nan, v) for k, v in PAIR_AUC.items()}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.15]})
for ax, (rep, auc) in zip(axes[:2], PAIR_AUC.items()):
    im = ax.imshow(auc, cmap="RdYlGn", vmin=0.5, vmax=1.0)
    for i in range(5):
        for j in range(5):
            if i != j:
                ax.text(j, i, f"{auc[i, j]:.2f}", ha="center", va="center", fontsize=10,
                        fontweight="bold" if abs(i - j) == 1 else "normal")
    ax.set_xticks(range(5), GRADES)
    ax.set_yticks(range(5), GRADES)
    ax.set_title(f"Pairwise test ROC-AUC – {rep} features\n(bold = adjacent grades)", fontsize=10)
fig.colorbar(im, ax=axes[:2], fraction=0.025)
for rep, auc in PAIR_AUC.items():
    dist_auc = [np.nanmean([auc[i, i + d] for i in range(5 - d)]) for d in range(1, 5)]
    axes[2].plot(range(1, 5), dist_auc, "o-", lw=2, label=rep)
    for d, v in zip(range(1, 5), dist_auc):
        axes[2].annotate(f"{v:.2f}", (d, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
axes[2].axhline(0.5, color="grey", ls=":", label="indistinguishable (0.5)")
axes[2].set_xticks(range(1, 5), ["1 grade\n(e.g. KL0–KL1)", "2 grades", "3 grades", "4 grades\n(KL0–KL4)"])
axes[2].set_ylim(0.45, 1.02)
axes[2].set_ylabel("Mean ROC-AUC")
axes[2].set_title("Separability vs distance between grades", fontsize=10)
axes[2].legend(fontsize=8)
plt.savefig(RESULTS_DIR / "pairwise_separability.png", dpi=200, bbox_inches="tight")
plt.show()

for rep, auc in PAIR_AUC.items():
    adj = [auc[i, i + 1] for i in range(4)]
    print(f"{rep}: adjacent pairs AUC = " + ", ".join(f"{GRADES[i]}–{GRADES[i + 1]} {a:.2f}" for i, a in enumerate(adj))
          + f" | KL0–KL4 AUC = {auc[0, 4]:.2f}")

### 8.2 Visual similarity: the average knee of each grade
Averaging all training X-rays of a grade removes patient-specific detail and leaves what is typical for that grade. The bottom row shows the **difference between neighbouring grades** (red = brighter in the higher grade, blue = darker). The numbers give the mean absolute pixel difference and the structural similarity (SSIM) between neighbouring average images.

If neighbouring grades are visually almost identical, the differences will be small and confined to the joint line – exactly the region where a few pixels of joint-space narrowing or a small osteophyte decide the grade.

In [ ]:
def ssim(a: np.ndarray, b: np.ndarray) -> float:
    """Structural similarity (global, Gaussian-weighted) between two grayscale images in [0, 255]."""
    a, b = a.astype(np.float64), b.astype(np.float64)
    c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
    blur = lambda x: cv2.GaussianBlur(x, (11, 11), 1.5)
    mu_a, mu_b = blur(a), blur(b)
    s_a, s_b, s_ab = blur(a * a) - mu_a ** 2, blur(b * b) - mu_b ** 2, blur(a * b) - mu_a * mu_b
    return float((((2 * mu_a * mu_b + c1) * (2 * s_ab + c2)) / ((mu_a ** 2 + mu_b ** 2 + c1) * (s_a + s_b + c2))).mean())


train_imgs = split_images("train").astype(np.float32)
mean_img = np.stack([train_imgs[GRADES_OF["train"] == g].mean(0) for g in range(5)])
fig, axes = plt.subplots(2, 5, figsize=(16, 7.8), gridspec_kw={"hspace": 0.3})
for g in range(5):
    axes[0, g].imshow(mean_img[g], cmap="gray", vmin=0, vmax=255)
    axes[0, g].set_title(f"Average {GRADES[g]} (n = {(GRADES_OF['train'] == g).sum():,})", fontsize=10)
    axes[0, g].axis("off")
sim_rows = []
for g in range(4):
    diff = mean_img[g + 1] - mean_img[g]
    axes[1, g].imshow(diff, cmap="seismic", vmin=-25, vmax=25)
    mad, s = np.abs(diff).mean(), ssim(mean_img[g], mean_img[g + 1])
    sim_rows.append({"Pair": f"{GRADES[g]} → {GRADES[g + 1]}", "Mean abs. difference (grey levels)": mad, "SSIM": s})
    axes[1, g].set_title(f"{GRADES[g + 1]} − {GRADES[g]}\n|diff| {mad:.1f} · SSIM {s:.3f}", fontsize=10)
    axes[1, g].axis("off")
far = mean_img[4] - mean_img[0]
axes[1, 4].imshow(far, cmap="seismic", vmin=-25, vmax=25)
sim_rows.append({"Pair": "KL0 → KL4 (far apart)", "Mean abs. difference (grey levels)": np.abs(far).mean(),
                 "SSIM": ssim(mean_img[0], mean_img[4])})
axes[1, 4].set_title(f"KL4 − KL0 (for contrast)\n|diff| {np.abs(far).mean():.1f} · SSIM {sim_rows[-1]['SSIM']:.3f}", fontsize=10)
axes[1, 4].axis("off")
plt.suptitle("Average training X-ray per grade (top) and differences between grades (bottom; red = brighter, blue = darker)", y=1.0)
plt.savefig(RESULTS_DIR / "average_knee_per_grade.png", dpi=200, bbox_inches="tight")
plt.show()
display(pd.DataFrame(sim_rows).style.format({"Mean abs. difference (grey levels)": "{:.2f}", "SSIM": "{:.4f}"}).hide(axis="index"))

### 8.3 How the grades are arranged in feature space
A t-SNE map places every test X-ray as a point so that images with similar DenseNet121 features lie close together. If the grades were visually distinct, five separate clusters would appear. If instead the grades are **mixed together**, the features do not separate the grades by themselves – what a gradual, ordinal disease with subtle differences looks like.

Look also at what *does* form clusters: groups that are not explained by grade reflect other image properties (e.g. exposure, contrast or positioning). They show that frozen ImageNet features describe general image appearance more strongly than the subtle signs of OA – a second reason, besides grade similarity, why the five-grade accuracy is limited.

In [ ]:
tsne_file = RESULTS_DIR / "tsne_test_densenet.npy"
if REUSE_SAVED_RESULTS and tsne_file.exists():
    emb = np.load(tsne_file)
else:
    Xs = StandardScaler().fit_transform(FEATURES["DenseNet121"]["test"])
    emb = TSNE(n_components=2, perplexity=40, init="pca", random_state=SEED).fit_transform(Xs)
    np.save(tsne_file, emb)

colours = ["#4C9F70", "#A3C45A", "#E3B23C", "#E07B39", "#C0392B"]
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for g in range(5):
    m = GRADES_OF["test"] == g
    axes[0].scatter(emb[m, 0], emb[m, 1], s=9, alpha=0.65, color=colours[g], label=f"{GRADES[g]} (n={m.sum()})")
axes[0].set_title("Test X-rays in DenseNet121 feature space (t-SNE), coloured by grade")
axes[0].legend(markerscale=2.5, fontsize=9)
for cls, colour in zip([0, 1], ["#4C9F70", "#C0392B"]):
    m = TASK_OBJ["A"](GRADES_OF["test"]) == cls
    axes[1].scatter(emb[m, 0], emb[m, 1], s=9, alpha=0.55, color=colour, label=TASK_OBJ["A"].classes[cls])
axes[1].set_title("Same map, coloured by task A (KL0 vs KL1–4)")
axes[1].legend(markerscale=2.5, fontsize=9)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.savefig(RESULTS_DIR / "tsne_grades.png", dpi=200)
plt.show()

---
## 9. Conclusion
The cell below prints the key numbers of this run; the interpretation that follows explains how they answer the question.

In [ ]:
print("KEY NUMBERS (test set)\n")
for task in TASK_OBJ.values():
    best_id = max([BEST[(task.title, m)] for m in model_order], key=lambda e: RESULTS[e]["val"]["macro_f1"])
    b = RESULTS[best_id]["test"]
    print(f"Task {task.key} ({task.title}) – best model on validation: {RESULTS[best_id]['model']} ({RESULTS[best_id]['imbalance']})")
    print(f"   accuracy {b['accuracy']:.3f} (majority baseline {RESULTS[best_id]['majority_baseline_test_accuracy']:.3f}) | "
          f"balanced acc. {b['balanced_accuracy']:.3f} | sensitivity {b['sensitivity']:.3f} | specificity {b['specificity']:.3f} | "
          f"AUC {b['roc_auc']:.3f} | κ {b['kappa']:.3f}")
if FIVE:
    best5 = max(FIVE.values(), key=lambda r: r["val"]["macro_f1"])
    p5 = np.asarray(best5["y_pred_test"])
    print(f"\nFive-class best model ({best5['id']}, {best5['family']}): accuracy {accuracy_score(GRADES_OF['test'], p5):.3f} | "
          f"κ {cohen_kappa_score(GRADES_OF['test'], p5):.3f} | one-grade-off share of errors "
          f"{(np.abs(p5 - GRADES_OF['test'])[p5 != GRADES_OF['test']] == 1).mean():.0%}")
for rep, auc in PAIR_AUC.items():
    print(f"Pairwise AUC ({rep}): mean adjacent {np.nanmean([auc[i, i + 1] for i in range(4)]):.2f} vs KL0–KL4 {auc[0, 4]:.2f}")
print(f"Average-image SSIM: adjacent grades {np.mean([r['SSIM'] for r in sim_rows[:4]]):.3f} vs KL0–KL4 {sim_rows[-1]['SSIM']:.3f}")

### Interpretation
*(Ranges below are from the full-dataset run used to write this notebook; the SVM and Random Forest give identical numbers on re-runs, the neural models can differ in the second decimal.)*

**1. Merging grades raises accuracy clearly – partly because the question is easier, partly because chance is higher.** Test accuracy rises from ≈0.47–0.51 (five grades) to ≈0.70–0.72 for task A and ≈0.74–0.77 for task B, well above the majority-class accuracy of each task (0.39, 0.61 and 0.57). The chance-corrected agreement **κ also rises** (≈0.27–0.30 → ≈0.38–0.43 → ≈0.45–0.53), so the gain is not only an effect of having fewer classes. Balanced accuracy *above chance*, however, stays at a similar level (≈0.2–0.26) in all settings: the models do not suddenly "see" more in the image – the binary question simply removes most of the fine boundaries they could not draw.

**2. A large part of the five-class error was confusion between neighbouring grades.** 57–72 % of all five-class errors are exactly one grade off, and 36–46 % (task A) to 44–54 % (task B) of them disappear when the five-grade predictions are merged after prediction. The merged five-class predictions reach almost the same accuracy and κ as models trained directly on the binary task (differences ≈0.04 or less): the five-grade models already rank knees by severity correctly at a coarse level; what they cannot do reliably is put a knee on the exact grade.

**3. The remaining binary errors sit at the boundary and move with it.** In task A the worst-classified grade is **KL1** (only ≈41–63 % on the correct side), while KL3 and KL4 are classified as OA in ≈87–100 % of cases. When KL1 is moved to the healthy side (task B), KL1 improves to ≈67–77 % and the worst grade becomes **KL2** (≈45–59 %), the new neighbour of the boundary. The errors therefore follow the boundary between neighbouring grades, not a particular model or a particular grade.

**4. Separability depends on the distance between grades.** With the same features the models use, adjacent grades are only modestly separable (pairwise ROC-AUC ≈0.59–0.89; the hardest pair is KL0 vs KL1 at ≈0.59–0.64), grades two apart reach ≈0.89–0.90, and KL0 vs KL4 is separated almost perfectly (AUC 1.00). The early grades are the least separable, which matches the definition of KL1 as "doubtful". The average X-ray of neighbouring grades differs by only ≈1.5–3.7 grey levels out of 255 (SSIM ≥ 0.996), and the difference is concentrated along the **joint line** – exactly where a few pixels of joint-space narrowing or a small osteophyte decide the grade.

**5. A second factor: what the frozen features pay attention to.** In the t-SNE map the grades are intermixed, while the largest clusters are not related to grade at all. Frozen ImageNet features describe general image appearance more strongly than the subtle signs of OA, so better features (fine-tuning on knee X-rays, cropping the joint region) would be needed to separate neighbouring grades further.

**Answer to the question.** Yes – the evidence supports the explanation. The relatively low five-class accuracy of the main notebook is largely caused by the **ordinal, gradual nature of KL grading**: neighbouring grades produce nearly identical X-rays, the early grades (KL0/KL1/KL2) are the least separable, and the errors always concentrate at whichever grades meet at a class boundary. When fewer boundaries have to be drawn, the same images and the same models become clearly more accurate. Two honest limits: (i) part of the accuracy gain comes from the higher chance level of a two-class task, so κ and the pairwise analysis are the stronger evidence; (ii) the binary task is still far from perfect (≈70–77 %), because its boundary also passes between two similar neighbouring grades and because frozen ImageNet features are not specialised for knee X-rays.

**For the report:** cite this notebook in the Results & Discussion section as a *control experiment*. Suggested figures: the five-class vs binary accuracy and κ chart (`five_vs_binary.png`), the errors-by-grade chart (`binary_errors_by_grade.png`), the pairwise separability heat-maps (`pairwise_separability.png`) and the average-knee comparison (`average_knee_per_grade.png`).